# 05 — Reproduction des figures du papier

Reproduit les analyses de la section **Usage Notes** (Nsumba et al., 2026) :

| Figure du papier | Contenu |
|---|---|
| Fig. 8 | Carte SPL moyen (Entebbe, avr–mai 2023) |
| Fig. 9 | Résumé horaire : médiane + IQR sur 24 h |
| Fig. 10 | Comparaison jour vs nuit |
| Discussion | Corrélation morphologie ↔ SPL (déjà fait au notebook 04) |

Quand vous aurez les mesures Hanoï, ces mêmes cellules produiront les figures Hanoï
pour la comparaison côte à côte dans le papier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap

df = pd.read_csv('../data/processed/sunbird_clean.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
# Définition jour/nuit standard OMS : nuit = 22h-6h
df['period'] = np.where((df['hour'] >= 22) | (df['hour'] < 6), 'Night', 'Day')
print(f'{len(df)} mesures')

## Fig. 8 — Carte SPL moyen (heatmap spatiale)

In [ ]:
# Agrège par cellule de grille (~100 m) puis heatmap du SPL moyen
GRID = 0.001  # ~100 m
df['lat_bin'] = (df['latitude'] / GRID).round() * GRID
df['lon_bin'] = (df['longitude'] / GRID).round() * GRID
cells = df.groupby(['lat_bin', 'lon_bin'])['noise_measurement'].mean().reset_index()

m = folium.Map(location=[df.latitude.mean(), df.longitude.mean()],
               zoom_start=12, tiles='CartoDB positron')
HeatMap(
    data=cells[['lat_bin', 'lon_bin', 'noise_measurement']].values.tolist(),
    radius=12, blur=8, min_opacity=0.4
).add_to(m)
m.save('../outputs/maps/fig8_mean_spl_map.html')
print(f'{len(cells)} cellules — carte → outputs/maps/fig8_mean_spl_map.html')
m

## Fig. 9 — Cycle horaire : médiane + IQR

Le papier observe : médiane diurne 45–50 dB, matins calmes, pic vers 21h–22h.

In [ ]:
hourly = df.groupby('hour')['noise_measurement'].agg(
    median='median',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
    n='count'
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hourly.index, hourly['median'], marker='o', color='steelblue', label='Médiane')
ax.fill_between(hourly.index, hourly['q25'], hourly['q75'],
                alpha=0.25, color='steelblue', label='IQR (25–75%)')
ax.set_xlabel('Heure de la journée')
ax.set_ylabel('SPL (dB)')
ax.set_xticks(range(0, 24))
ax.set_title('Cycle horaire du SPL — reproduction Fig. 9')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/maps/fig9_hourly_cycle.png', dpi=150)
plt.show()

print('Vérifs vs papier :')
print(f"  Médiane diurne (8h-18h) : {df[df.hour.between(8,18)]['noise_measurement'].median():.1f} dB (papier : 45-50)")
print(f"  Heure du pic : {hourly['median'].idxmax()}h (papier : 21h-22h)")

## Fig. 10 — Jour vs Nuit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for period, color in [('Day', 'goldenrod'), ('Night', 'navy')]:
    sub = df[df['period'] == period]['noise_measurement']
    axes[0].hist(sub, bins=35, alpha=0.55, label=f'{period} (n={len(sub)})',
                 color=color, density=True)
axes[0].set_xlabel('SPL (dB)')
axes[0].set_title('Distributions jour vs nuit')
axes[0].legend()

sns.boxplot(data=df, x='period', y='noise_measurement', ax=axes[1],
            palette={'Day': 'goldenrod', 'Night': 'navy'})
axes[1].set_title('Jour vs Nuit — reproduction Fig. 10')

plt.tight_layout()
plt.savefig('../outputs/maps/fig10_day_night.png', dpi=150)
plt.show()

print(df.groupby('period')['noise_measurement'].describe().round(1))

## Checklist de reproduction

| Élément du papier | Notebook | Statut |
|---|---|---|
| Nettoyage (doublons, GPS, dB) | 02 | fait |
| QC audio (RMS, spectre, hash) | 03 | fait |
| Morphologie urbaine (R=300m) | 04 | fait |
| Carte SPL moyen (Fig. 8) | 05 | fait |
| Cycle horaire (Fig. 9) | 05 | fait |
| Jour vs nuit (Fig. 10) | 05 | fait |
| Corrélation morphologie-SPL | 04 | fait |
| Calibration téléphone (Table 1) | terrain Hanoï | à faire — cross-calibration entre vos téléphones |
| Collecte ODK Collect | terrain Hanoï | à faire — même app que le papier |

**Note nuit** : le papier signale peu de données nocturnes (sécurité). Vérifier le `n` par heure
avant d'interpréter — même prudence pour vos mesures à minuit à Hanoï.